https://knaaptime.com/urban_analysis/02_interpolation/dasymetric.html

#### interpolation of resident numbers from 2010 to 2021

Using Target Density Weighting (TDW) method found in the paper `Data-enriched Interpolation for Temporally Consistent Population Compositions`

1. Calculate subzone densities:
$Density = \frac{residents\ in\ subzone}{land area}$

2. Establish weight ratio:

Assume the subzone population is proportional to the density of the year's total population. Calculate a `density ratio` for each subzone

$Ratio = \frac{Subzone\ Density}{National\ Average\ Density}$

3. Temporal Interpolation of Weights:

As we only have population data for 2010, 2015 and 2020, need to estimate the density weights for the missing years.

Target Density Weighting method assumes that the subzone's density ratio moves linearly over time.

4. Interpolate:

Multiply the annual national counts for each age group by the calculated weights. 

#### Interpolate resident numbers by age group
Census data for 2020 follows 2019 masterplan for geospatial data, 2015 follows 2014 masterplan, 2010 follows 2008 masterplan

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
from shapely.wkt import loads

In [3]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [4]:
# get subzone data with their corresponding areas in km^2
def get_subzone_data(year: int):
    """
    Args
    ------
    year: int
        year of the subzones you want (2019, 2014, 2008)

    Returns
    ------
    Subzone dataframe containing the subzone names, planning area names and areas of points of interest.
    """

    year = str(year)

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_boundary_{year}.gpkg")
    subzones_gpd = gpd.read_file(subzones_filepath)
    subzones_gpd = subzones_gpd.map(lambda s: s.lower() if isinstance(s, str) else s)
    # lower() the column names of subzones_gpd
    subzones_gpd.columns = subzones_gpd.columns.str.lower()    

    # there is no landuse data found for 2008 masterplan, 
    # so not able to merge with the subzone_classification file obtained from extracting_landuse_type.ipynb
    if year == "2008":
        return subzones_gpd

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    subzones_csv = pd.read_csv(subzones_filepath)
    subzones_csv = subzones_csv.map(lambda s: s.lower() if isinstance(s, str) else s)

    subzones_with_geom = subzones_csv.merge(
        subzones_gpd[["subzone_n", "geometry"]],
        on = "subzone_n",
        how = "left"
    )

    missing_count = subzones_with_geom[subzones_with_geom.columns[-1]].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} subzones did not find a match in the CSV.")

    return subzones_with_geom

In [5]:
def get_demographic_data(year: int):

    year = str(year)

    demographic_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/{year}_age_group_planning_area_subzone.xlsx")
    demographic_df = pd.read_excel(demographic_filepath, sheet_name = "subzone")

    demographic_df = demographic_df.rename(columns = {
        "subzone": "subzone_n",
        "planning_area": "pln_area_n"
    })

    # 2015 census data has different column names, standardise it
    if year == "2015":
        old_cols = [
            'total_0_4', 'total_5_9', 'total_10_14', 'total_15_19', 'total_20_24', 
            'total_25_29', 'total_30_34', 'total_35_39', 'total_40_44', 'total_45_49', 
            'total_50_54', 'total_55_59', 'total_60_64', 'total_65_69', 'total_70_74', 
            'total_75_79', 'total_80_84', 'total_85andover'
        ]

        mapping = {}
        for col in old_cols:
            # Remove 'total_' prefix and replace underscores with ' - '
            new_name = col.replace('total_', '').replace('_', ' - ')
            
            # Handle the special '85andover' case you mentioned
            if '85andover' in new_name:
                mapping[col] = '85 - 89' # Based on your specific request
            else:
                mapping[col] = new_name

        mapping["total_85andover"] = "85 & Over"

        demographic_df = demographic_df.rename(columns = mapping)

    demographic_df["pln_area_n"] = demographic_df["pln_area_n"].ffill()

    ## the subzones for punggol is named differently in the masterplan for 2010 and 2015, 
    # instead of eg: subzone 2, it is sz2, so renaming it
    if year == "2015" or year == "2010":
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 2") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz2"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 3") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz3"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 4") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz4"
        demographic_df.loc[
            (demographic_df["subzone_n"] == "subzone 5") &
            (demographic_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz5"

    return demographic_df

In [6]:
def convert_to_geodataframe(df):
    if not isinstance(df, gpd.GeoDataFrame):
        df = gpd.GeoDataFrame(df, geometry='geometry')

    # ensure the original data has the correct starting CRS (WGS84)
    # (Only needed if it isn't already set; usually .gpkg files have this metadata)
    if df.crs is None:
        df.set_crs(epsg=4326, inplace=True)

    # transform the entire GeoDataFrame to SVY21 (Meters)
    df_meters = df.to_crs(epsg=3414)

    return df_meters

# Step 1 : Calculate population density per subzone

In [7]:
subzone_df = get_subzone_data(2008)
subzone_df.head(3)

,objectid,subzone_no,subzone_n,subzone_c,ca_ind,pln_area_n,pln_area_c,region_n,region_c,inc_crc,fmel_upd_d,x_addr,y_addr,shape_leng,shape_area,geometry
0,1,1,woodlands regional centre,wdsz01,n,woodlands,wd,north region,nr,8877712517deec59,2016-05-11,22571.2094,46698.8735,3254.343275,5.956521e+05,"MULTIPOLYGON (((22629.486 46980.246, 22683.012..."
1,2,1,kranji,sksz01,n,sungei kadut,sk,north region,nr,de75f42ec0f86b2b,2016-05-11,19767.5129,46226.3625,7906.523525,3.654121e+06,"MULTIPOLYGON (((20329.947 47234.988, 20333.366..."
2,3,2,turf club,sksz02,n,sungei kadut,sk,north region,nr,5d555db858522781,2016-05-11,20234.1723,44507.1299,7677.244752,3.290816e+06,"MULTIPOLYGON (((21094.367 44815.121, 21099.854..."


In [8]:
# convert the Pandas DataFrame back to a GeoDataFrame
# We explicitly pass the geometry column
if not isinstance(subzone_df, gpd.GeoDataFrame):
    subzone_df = gpd.GeoDataFrame(subzone_df, geometry='geometry')

# ensure the original data has the correct starting CRS (WGS84)
# (Only needed if it isn't already set; usually .gpkg files have this metadata)
if subzone_df.crs is None:
    subzone_df.set_crs(epsg=4326, inplace=True)

# transform the entire GeoDataFrame to SVY21 (Meters)
subzone_df_meters = subzone_df.to_crs(epsg=3414)

# calculate area
subzone_df['area_sqm'] = subzone_df_meters.geometry.area
subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000

subzone_areas = subzone_df[['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', "geometry"]].copy()

# display(subzone_df[['subzone_n', 'area_sqkm']].head())
display(subzone_areas)

,subzone_n,pln_area_n,area_sqm,area_sqkm,geometry
0,woodlands regional centre,woodlands,5.956521e+05,0.595652,"MULTIPOLYGON (((22629.486 46980.246, 22683.012..."
1,kranji,sungei kadut,3.654121e+06,3.654121,"MULTIPOLYGON (((20329.947 47234.988, 20333.366..."
2,turf club,sungei kadut,3.290816e+06,3.290816,"MULTIPOLYGON (((21094.367 44815.121, 21099.854..."
3,nee soon,yishun,2.209311e+06,2.209311,"MULTIPOLYGON (((26860.555 43913.328, 26839.719..."
4,yishun west,yishun,1.517705e+06,1.517705,"MULTIPOLYGON (((28228.609 45792.488, 28223.039..."
...,...,...,...,...,...
306,sungei serangoon west,sengkang,1.572812e+06,1.572812,"MULTIPOLYGON (((37009.969 40867.361, 37011.087..."
307,sungei serangoon,hougang,1.333531e+06,1.333531,"MULTIPOLYGON (((36538.391 39761.352, 36517.439..."
308,hougang central,hougang,2.291220e+06,2.291220,"MULTIPOLYGON (((35948.803 40264.817, 35951.927..."
309,changi west,changi,4.849019e+06,4.849019,"MULTIPOLYGON (((45347.86 41116.219, 45350.666 ..."


In [9]:
demographic_df = get_demographic_data(2010)
demographic_df

,pln_area_n,subzone_n,total,0 - 4,5 - 9,10 - 14,15 - 19,20 - 24,25 - 29,30 - 34,35 - 39,40 - 44,45 - 49,50 - 54,55 - 59,60 - 64,65 & Over,total_above_60
0,total,total,3771721,194432,215675,244302,263750,247190,272639,298687,320024,309441,323459,303044,248696,191995,338387,530382
1,ang mo kio,total,179297,7967,8424,9335,10457,10656,13400,14502,14510,13525,14862,14605,13785,11868,21401,33269
2,ang mo kio,cheng san,30503,1334,1387,1428,1587,1720,2471,2691,2569,2365,2471,2461,2487,2099,3433,5532
3,ang mo kio,chong boon,29903,1254,1304,1429,1563,1713,2348,2469,2326,2241,2431,2388,2380,2137,3920,6057
4,ang mo kio,kebun bahru,25854,1089,1189,1327,1418,1460,1901,2076,2123,1968,2122,2078,1992,1752,3359,5111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,yishun,yishun central,1608,67,45,102,161,144,147,116,79,109,161,152,124,76,125,201
224,yishun,yishun east,32933,1729,2029,2385,2975,2579,2353,2476,2642,2755,3244,2725,2018,1302,1721,3023
225,yishun,yishun south,40551,1842,2133,2488,3252,3406,3228,3008,3150,3033,3830,3491,2849,2009,2832,4841
226,yishun,yishun west,60219,2920,3222,3644,3916,4614,4987,4685,4816,4993,5408,5157,4189,2904,4764,7668


#### calculate the density of each subzone
Density (resident per $km^2$) = $\frac{residents\ in\ subzone}{land\ area\ km^2}$

####  each set of census data has different recordings of age group numbers. I have chosen to preserve them instead of combining ages 65 and over. This is because granularity will help with the accuracy of the Target Density Weighting interpolation

In [10]:
def calculate_density(demographic_df, subzone_areas, year:int):
    
    year = str(year)

    if year == "2010":
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 & Over', 'total_above_60']
    
    if year == "2015":    
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 & Over', 'total_above_60']
        
    if year == "2020":
        age_grp_columns = ['subzone_n', 'pln_area_n', 'total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 - 89', '90 & over', 'total_above_60']
        
    demographic_df = demographic_df[age_grp_columns].copy()

    # standardize casing and remove hidden whitespace
    subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
    demographic_df['subzone_n'] = demographic_df['subzone_n'].astype(str).str.strip().str.lower()

    density_df = subzone_areas.merge(
        demographic_df,
        on = ["subzone_n", "pln_area_n"],
        how = "left"
    )
    
    for age_group in age_grp_columns[2:]:
        new_column_name = "density_" + age_group

        density_df[f"density_{age_group}"] =  density_df[age_group] / density_df["area_sqkm"]

    return density_df
    

In [11]:
density_df = calculate_density(demographic_df, subzone_areas, 2010)
# fill NaN results with 0 (0 population data for that subzone)
density_df = density_df.fillna(np.nan)

# save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2008/pop_density_2010.csv")
# density_df.to_csv(save_to_file)

In [12]:
# sanity check to see if merging worked
# checking if left join duplicated any subzone rows. 
# Note that some planning areas have the same subzone names. 
# And some subzone did not appear in the demographic dataset so their density columns will be 0 / NaN

def sanity_checks(density_df, subzone_areas, demographic_df):
    duplicate_mask = density_df.duplicated(subset=['subzone_n'], keep=False)

    # Get the unique names of those duplicates
    duplicate_names = density_df.loc[duplicate_mask, 'subzone_n'].unique()

    print("Planning areas with duplicates:")
    print(list(duplicate_names))

    ## checking if merging was successful. 
    # this checks which subzone and planning area pair from the demographic_df did not match with subzone_areas dataframe.
    subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
    # subzone_areas.to_csv("subzone_areas.csv")
    demographic_df['subzone_n'] = demographic_df['subzone_n'].astype(str).str.strip().str.lower()
    # demographic_df.to_csv("demographic_df.csv")

    check_df = subzone_areas[['subzone_n', 'pln_area_n']].merge(
        demographic_df[['subzone_n', 'pln_area_n']], 
        on=['subzone_n', 'pln_area_n'],
        how='outer', 
        indicator=True
    )

    failed_df = check_df[check_df['_merge'] == 'right_only'][['subzone_n', 'pln_area_n']]
    print(failed_df.to_string(index=False))


In [13]:
# ## checking if merging was successful. 
# # this checks which subzone and planning area pair from the demographic_df did not match with subzone_areas dataframe.
# subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
# # subzone_areas.to_csv("subzone_areas.csv")
# demographic_df['subzone_n'] = demographic_df['subzone_n'].astype(str).str.strip().str.lower()
# # demographic_df.to_csv("demographic_df.csv")

# check_df = subzone_areas[['subzone_n', 'pln_area_n']].merge(
#     demographic_df[['subzone_n', 'pln_area_n']], 
#     on=['subzone_n', 'pln_area_n'],
#     how='outer', 
#     indicator=True
# )

# failed_df = check_df[check_df['_merge'] == 'right_only'][['subzone_n', 'pln_area_n']]
# print(failed_df.to_string(index=False))

# Step 2: Calculate Density Ratio per subzone

#### First we need to obtain the national average density for each age group

#### Calculating National Average Density
$\frac{Sum\ total\ resident}{Sum\ total\ area\ of\ Singapore}$

In [14]:
def calculate_national_average_density(density_df, year: int):
    """
    return the density of each age group across whole of Singapore
    """

    year = str(year)

    if year == "2010":
        age_cols = ['total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 & Over', 'total_above_60']

    if year == "2015":    
        age_cols = ['total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 & Over', 'total_above_60']

    if year == "2020":
        age_cols = ['total', '0 - 4', '5 - 9', '10 - 14',
       '15 - 19', '20 - 24', '25 - 29', '30 - 34', '35 - 39', '40 - 44',
       '45 - 49', '50 - 54', '55 - 59', '60 - 64', '65 - 69', '70 - 74',
       '75 - 79', '80 - 84', '85 - 89', '90 & over', 'total_above_60']

    # density_cols = [col for col in density_df.columns if "density" in col.lower()]

    total_area_all_subzones = density_df['area_sqkm'].sum()

    aggregated_results = []

    for col in age_cols:
        total_resident_count = density_df[col].sum()
        avg_density = total_resident_count / total_area_all_subzones
        
        aggregated_results.append({
            'age_group': col,
            'national_avg_density': avg_density
        })

    national_density_df = pd.DataFrame(aggregated_results)

    return national_density_df

In [15]:
national_age_grp_density_df = calculate_national_average_density(density_df, 2010)
national_age_grp_density_df

,age_group,national_avg_density
0,total,4834.802283
1,0 - 4,249.281804
2,5 - 9,276.665744
3,10 - 14,313.466093
4,15 - 19,338.496253
5,20 - 24,316.852549
6,25 - 29,349.582713
7,30 - 34,382.687575
8,35 - 39,409.853907
9,40 - 44,396.291343


#### Calculate Density Ratio of each subzone
$Ratio = \frac{Subzone\ Density}{National\ Average\ Density}$

In [16]:
def calculate_density_ratio(density_df, national_age_grp_density_df):
    age_group_list = national_age_grp_density_df["age_group"].tolist()
    
    for age_group in age_group_list:

        national_average_density = national_age_grp_density_df.loc[
            national_age_grp_density_df["age_group"] == age_group,
            "national_avg_density"
        ].item()

        # if that age group is 0, set to np.nan to indicate that the national average was 0. 
        if national_average_density <= 0:
            density_df[f"density_ratio_{age_group}"] = np.nan

        else:
            density_df[f"density_ratio_{age_group}"] = density_df[f"density_{age_group}"] / national_average_density

    return density_df

In [17]:
density_ratio_df = calculate_density_ratio(density_df, national_age_grp_density_df)

# export and check 
# save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_2008/pop_density_ratio_2010.csv")
# density_ratio_df.to_csv(save_to_file)

# 3. Temporal Interpolation of Weights

In [18]:
read_from_file_2010 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/pop_density_ratio_2010.csv")
read_from_file_2015 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/pop_density_ratio_2015.csv")
read_from_file_2020 = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/pop_density_ratio_2020.csv")

density_ratio_df_2010 = pd.read_csv(read_from_file_2010)
age_groups = [col for col in density_ratio_df_2010.columns if "density_ratio_" in col.lower()]
# display(age_groups)
density_ratio_df_2015 = pd.read_csv(read_from_file_2015)
age_groups = [col for col in density_ratio_df_2015.columns if "density_ratio_" in col.lower()]
# display(age_groups)
density_ratio_df_2020 = pd.read_csv(read_from_file_2020)
age_groups = [col for col in density_ratio_df_2020.columns if "density_ratio_" in col.lower()]
display(age_groups)

['density_ratio_total',
 'density_ratio_0 - 4',
 'density_ratio_5 - 9',
 'density_ratio_10 - 14',
 'density_ratio_15 - 19',
 'density_ratio_20 - 24',
 'density_ratio_25 - 29',
 'density_ratio_30 - 34',
 'density_ratio_35 - 39',
 'density_ratio_40 - 44',
 'density_ratio_45 - 49',
 'density_ratio_50 - 54',
 'density_ratio_55 - 59',
 'density_ratio_60 - 64',
 'density_ratio_65 - 69',
 'density_ratio_70 - 74',
 'density_ratio_75 - 79',
 'density_ratio_80 - 84',
 'density_ratio_85 - 89',
 'density_ratio_90 & over',
 'density_ratio_total_above_60']

In [19]:
# # plot the density ratio changes for each planning area to check if there is any sudden spike or decrease in density ratio
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import os

# # 1. Prepare the temporal data (Long Format)
# # We assume the columns are named 'subzone_n', 'pln_area_n', and 'density_ratio_total'
# d10 = density_ratio_df_2010[['subzone_n', 'pln_area_n', 'density_ratio_total']].rename(columns={'density_ratio_total': '2010'})
# d15 = density_ratio_df_2015[['subzone_n', 'pln_area_n', 'density_ratio_total']].rename(columns={'density_ratio_total': '2015'})
# d20 = density_ratio_df_2020[['subzone_n', 'pln_area_n', 'density_ratio_total']].rename(columns={'density_ratio_total': '2020'})

# # Merge and Melt
# merged = d10.merge(d15, on=['subzone_n', 'pln_area_n'], how='outer')
# merged = merged.merge(d20, on=['subzone_n', 'pln_area_n'], how='outer')
# df_long = merged.melt(id_vars=['subzone_n', 'pln_area_n'], var_name='Year', value_name='Ratio')
# df_long['Year'] = df_long['Year'].astype(int)
# df_long = df_long.fillna(0)

# # save_to_file = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/consolidated_data.csv")
# # df_long.to_csv(save_to_file)

# # Create a directory to store the images if it doesn't exist
# output_dir = "planning_area_plots"
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)

# # 2. Loop through unique Planning Areas
# unique_areas = df_long['pln_area_n'].unique()

# for area in unique_areas:
#     # Filter data for this specific area
#     subset = df_long[df_long['pln_area_n'] == area]
    
#     # Skip if area is NaN or empty
#     if pd.isna(area) or subset.empty:
#         continue

#     # Create the plot
#     plt.figure(figsize=(12, 7))
#     sns.lineplot(data=subset, x='Year', y='Ratio', hue='subzone_n', marker='o', linewidth=2)
    
#     # Formatting
#     plt.title(f"Temporal Density Ratio: {area.upper()}", fontsize=15)
#     plt.ylabel("Density Ratio (Subzone / National)")
#     plt.xlabel("Census Year")
#     plt.xticks([2010, 2015, 2020])
#     plt.ylim(0, subset['Ratio'].max() * 1.2) # Adjust height based on local max
    
#     # Legend handling (move outside if many subzones)
#     plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Subzones")
#     plt.grid(True, linestyle=':', alpha=0.6)
    
#     # Save the file
#     # Replace spaces with underscores for cleaner filenames
#     filename = f"{area.replace(' ', '_').lower()}_trend.png"
#     plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight', dpi=150)
    
#     # Close the plot to free up memory during the loop
#     plt.close()

# print(f"Successfully saved {len(unique_areas)} plots to the '{output_dir}' folder.")

As we only have population data for 2010, 2015 and 2020, need to estimate the density weights for the missing years.

Target Density Weighting method assumes that the subzone's density ratio moves linearly over time. So to interpolate for the density ratio changes from 2011 to 2020, we draw a stright line between the 2010 - 2015 density ratio and 2015 - 2020 density ratio. 

In [20]:
# vertical concatenation of dataframes
base_ids = ['subzone_n', 'pln_area_n']

# add Year tags and filter for your specific columns
# 2010 Data
cols_2010 = base_ids + [col for col in density_ratio_df_2010.columns if "density_ratio_" in col.lower()]
d10 = density_ratio_df_2010[cols_2010].copy()
d10['Year'] = 2010

# 2015 Data
cols_2015 = base_ids + [col for col in density_ratio_df_2015.columns if "density_ratio_" in col.lower()]
d15 = density_ratio_df_2015[cols_2015].copy()
d15['Year'] = 2015

# 2020 Data
cols_2020 = base_ids + [col for col in density_ratio_df_2020.columns if "density_ratio_" in col.lower()]
d20 = density_ratio_df_2020[cols_2020].copy()
d20['Year'] = 2020

# pd.concat will automatically handle the differing columns by adding NaNs where data is missing
df_stacked = pd.concat([d10, d15, d20], axis=0, ignore_index=True, sort=False)

# Melt into Long Format
# This will create a single "AgeGroup" column and a "WeightingRatio" column
df_long = df_stacked.melt(
    id_vars=['subzone_n', 'pln_area_n', 'Year'],
    var_name='AgeGroup_Ratio',
    value_name='WeightingRatio'
)

# remove 'density_ratio_' prefix)
df_long['AgeGroup'] = df_long['AgeGroup_Ratio'].str.replace('density_ratio_', '')
df_long = df_long.drop(columns = ["AgeGroup_Ratio"])

# save_to_file = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/consolidated_data.csv")
# df_long.to_csv(save_to_file)

display(df_long)


,subzone_n,pln_area_n,Year,WeightingRatio,AgeGroup
0,woodlands regional centre,woodlands,2010,NaN,total
1,kranji,sungei kadut,2010,NaN,total
2,turf club,sungei kadut,2010,NaN,total
3,nee soon,yishun,2010,NaN,total
4,yishun west,yishun,2010,8.206681,total
...,...,...,...,...,...
22213,springleaf,yishun,2020,0.337834,90 & over
22214,yishun central,yishun,2020,0.855180,90 & over
22215,yishun east,yishun,2020,2.531068,90 & over
22216,yishun south,yishun,2020,4.733698,90 & over


In [21]:
# import pandas as pd

# # create a split of density ratio for age groups not recorded in 2010 and 2015 data.
# # 2010 is missing ['65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over'] age groups
# # 2015 is missing ['85 - 89', '90 & over'] age groups
# def split_consolidated_groups(df_long):
#     # --- 1. Calculate 2015 Distribution Shares (for 2010 split) ---
#     cols_2015_split = ['65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over']
#     df_2015 = df_long[df_long['Year'] == 2015].copy()
    
#     sum_2015 = df_2015[df_2015['AgeGroup'].isin(cols_2015_split)].groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].sum().reset_index(name='Sum_65p_2015')
    
#     shares_2015 = df_2015[df_2015['AgeGroup'].isin(cols_2015_split)].merge(sum_2015, on=['subzone_n', 'pln_area_n'])
#     shares_2015['share'] = shares_2015['WeightingRatio'] / shares_2015['Sum_65p_2015']
#     shares_2015['share'] = shares_2015['share'].fillna(0)

#     # --- 2. Create Synthetic 2010 Data ---
#     df_2010_total = df_long[(df_long['Year'] == 2010) & (df_long['AgeGroup'] == '65 & Over')][['subzone_n', 'pln_area_n', 'WeightingRatio']]
#     df_2010_total.rename(columns={'WeightingRatio': 'Total_2010_Val'}, inplace=True)
    
#     synthetic_2010 = shares_2015.merge(df_2010_total, on=['subzone_n', 'pln_area_n'])
#     synthetic_2010['WeightingRatio'] = synthetic_2010['Total_2010_Val'] * synthetic_2010['share']
#     synthetic_2010['Year'] = 2010
    
#     # --- 3. Create Synthetic 2015 Data using 2020 distribution ---
#     cols_2020_split = ['85 - 89', '90 & over']
#     df_2020 = df_long[df_long['Year'] == 2020].copy()
#     sum_2020 = df_2020[df_2020['AgeGroup'].isin(cols_2020_split)].groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].sum().reset_index(name='Sum_85p_2020')
    
#     shares_2020 = df_2020[df_2020['AgeGroup'].isin(cols_2020_split)].merge(sum_2020, on=['subzone_n', 'pln_area_n'])
#     shares_2020['share'] = shares_2020['WeightingRatio'] / shares_2020['Sum_85p_2020']
#     shares_2020['share'] = shares_2020['share'].fillna(0)
    
#     df_2015_85_total = df_long[(df_long['Year'] == 2015) & (df_long['AgeGroup'] == '85 & Over')][['subzone_n', 'pln_area_n', 'WeightingRatio']]
#     df_2015_85_total.rename(columns={'WeightingRatio': 'Total_2015_85_Val'}, inplace=True)
    
#     synthetic_2015 = shares_2020.merge(df_2015_85_total, on=['subzone_n', 'pln_area_n'])
#     synthetic_2015['WeightingRatio'] = synthetic_2015['Total_2015_85_Val'] * synthetic_2015['share']
#     synthetic_2015['Year'] = 2015

#     # --- 4. CRITICAL CLEANUP: Drop the original empty/consolidated rows ---
#     # Drop rows that were originally 0/NaN placeholders for the groups we just synthesized
#     cleaned_original = df_long[
#         ~((df_long['Year'] == 2010) & (df_long['AgeGroup'].isin(cols_2015_split))) & # Remove the 0s in 2010
#         ~((df_long['Year'] == 2015) & (df_long['AgeGroup'].isin(cols_2020_split))) & # Remove the 0s in 2015
#         ~((df_long['AgeGroup'] == '65 & Over')) &                                  # Remove the source consolidated labels
#         ~((df_long['AgeGroup'] == '85 & Over') & (df_long['Year'] == 2015))        # Remove 2015 consolidated 85+
#     ].copy()

#     cols_to_keep = ['subzone_n', 'pln_area_n', 'Year', 'AgeGroup', 'WeightingRatio']
#     final_df = pd.concat([cleaned_original[cols_to_keep], 
#                           synthetic_2010[cols_to_keep], 
#                           synthetic_2015[cols_to_keep]], ignore_index=True)
    
#     return final_df

# split_senior_age_group = split_consolidated_groups(df_long)

# save_to_file = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/split_consolidated_data.csv")
# split_senior_age_group.to_csv(save_to_file)

# split_senior_age_group

In [ ]:
# import pandas as pd
# import numpy as np

# def temporal_interpolation(df_long):
#     # --- STEP 1: Aggregate 2015 & 2020 to match older anchors ---
    
#     # A. Aggregate 2015 to create a '65 & Over' total to match 2010
#     cols_65p = ['65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over']
#     df_2015_agg = df_long[(df_long['Year'] == 2015) & (df_long['AgeGroup'].isin(cols_65p))]
#     agg_2015 = df_2015_agg.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].sum().reset_index()
#     agg_2015['AgeGroup'] = '65 & Over'
#     agg_2015['Year'] = 2015

#     # B. Aggregate 2020 to create an '85 & Over' total to match 2015
#     cols_85p = ['85 - 89', '90 & over']
#     df_2020_agg = df_long[(df_long['Year'] == 2020) & (df_long['AgeGroup'].isin(cols_85p))]
#     agg_2020 = df_2020_agg.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].sum().reset_index()
#     agg_2020['AgeGroup'] = '85 & Over'
#     agg_2020['Year'] = 2020

#     # Combine these new aggregated points back into the main dataset
#     df_combined = pd.concat([df_long, agg_2015, agg_2020], ignore_index=True)

#     # --- STEP 2: Interpolation backbone (2010-2020) ---
#     years = np.arange(2010, 2021)
#     subzones = df_combined[['subzone_n', 'pln_area_n']].drop_duplicates()
    
#     # We define the specific labels we want to interpolate
#     target_labels = ['65 & Over', '85 & Over']
    
#     df_timeline = subzones.assign(k=1).merge(pd.DataFrame({'Year': years, 'k': 1}), on='k')
#     df_timeline = df_timeline.assign(k=1).merge(pd.DataFrame({'AgeGroup': target_labels, 'k': 1}), on='k').drop('k', axis=1)

#     # Merge and Interpolate
#     final_interp = pd.merge(df_timeline, df_combined, on=['subzone_n', 'pln_area_n', 'Year', 'AgeGroup'], how='left')
#     final_interp = final_interp.sort_values(['subzone_n', 'pln_area_n', 'AgeGroup', 'Year'])
    
#     # Linear interpolation between 2010 and 2015 for '65 & Over'
#     # And 2015 to 2020 for '85 & Over'
#     final_interp['WeightingRatio'] = final_interp.groupby(['subzone_n', 'pln_area_n', 'AgeGroup'])['WeightingRatio'].transform(
#         lambda x: x.interpolate(method='linear', limit_direction='both')
#     )
    
#     return final_interp

# # Run the simple interpolation
# df_interpolated_seniors = temporal_interpolation(df_long)

# save_to_file = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/temporal_interpolation.csv")
# df_interpolated_seniors.to_csv(save_to_file)

# df_interpolated_seniors

,subzone_n,pln_area_n,Year,AgeGroup,WeightingRatio
450,admiralty,sembawang,2010,65 & Over,1.355293
452,admiralty,sembawang,2011,65 & Over,2.385029
454,admiralty,sembawang,2012,65 & Over,3.414765
456,admiralty,sembawang,2013,65 & Over,4.444501
458,admiralty,sembawang,2014,65 & Over,5.474237
...,...,...,...,...,...
3001,yunnan,jurong west,2017,85 & Over,3.830276
3003,yunnan,jurong west,2018,85 & Over,4.429311
3005,yunnan,jurong west,2019,85 & Over,5.028347
3007,yunnan,jurong west,2020,85 & Over,5.627382


In [ ]:
# NOT BEING USED, CONTAINS ERRORS

# def temporal_interpolation_with_decay(df_long):
#     # 1. Identify all unique subzones across the entire dataset
#     subzones = df_long[['subzone_n', 'pln_area_n']].drop_duplicates()
    
#     # --- Segment A: 2010 to 2015 (65 & Over) ---
#     # Prepare 2010 anchors
#     a10 = df_long[(df_long['Year'] == 2010) & (df_long['AgeGroup'] == '65 & Over')]
    
#     # Prepare 2015 aggregated anchors
#     cols_65p = ['65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over']
#     a15_65p = df_long[(df_long['Year'] == 2015) & (df_long['AgeGroup'].isin(cols_65p))].copy()
#     a15_65p = a15_65p.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].sum().reset_index()
#     a15_65p['AgeGroup'], a15_65p['Year'] = '65 & Over', 2015

#     # Create Bridge 1 backbone
#     bridge_1 = subzones.assign(k=1).merge(pd.DataFrame({'Year': np.arange(2010, 2016), 'k': 1}), on='k').drop('k', axis=1)
#     bridge_1['AgeGroup'] = '65 & Over'
    
#     # Merge anchors and FILL NA WITH 0 for the anchors only
#     # This ensures eg: Pasir Ris has a '0' in 2015 to trend toward
#     anchors_1 = pd.concat([a10, a15_65p])
#     bridge_1 = bridge_1.merge(anchors_1, on=['subzone_n', 'pln_area_n', 'Year', 'AgeGroup'], how='left')
    
#     # Force 0s at the anchor years if they are missing
#     bridge_1.loc[bridge_1['Year'].isin([2010, 2015]), 'WeightingRatio'] = bridge_1.loc[bridge_1['Year'].isin([2010, 2015]), 'WeightingRatio'].fillna(0)
    
#     bridge_1['WeightingRatio'] = bridge_1.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].transform(lambda x: x.interpolate(method='linear'))

#     # --- Segment B: 2015 to 2020 (85 & Over) ---
#     a15_85p = df_long[(df_long['Year'] == 2015) & (df_long['AgeGroup'] == '85 & Over')]
    
#     cols_85p = ['85 - 89', '90 & over']
#     a20_85p = df_long[(df_long['Year'] == 2020) & (df_long['AgeGroup'].isin(cols_85p))].copy()
#     a20_85p = a20_85p.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].sum().reset_index()
#     a20_85p['AgeGroup'], a20_85p['Year'] = '85 & Over', 2020

#     # Create Bridge 2 backbone
#     bridge_2 = subzones.assign(k=1).merge(pd.DataFrame({'Year': np.arange(2015, 2021), 'k': 1}), on='k').drop('k', axis=1)
#     bridge_2['AgeGroup'] = '85 & Over'
    
#     anchors_2 = pd.concat([a15_85p, a20_85p])
#     bridge_2 = bridge_2.merge(anchors_2, on=['subzone_n', 'pln_area_n', 'Year', 'AgeGroup'], how='left')
    
#     # Force 0s at 2015 and 2020 anchors
#     bridge_2.loc[bridge_2['Year'].isin([2015, 2020]), 'WeightingRatio'] = bridge_2.loc[bridge_2['Year'].isin([2015, 2020]), 'WeightingRatio'].fillna(0)
    
#     bridge_2['WeightingRatio'] = bridge_2.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].transform(lambda x: x.interpolate(method='linear'))

#     return pd.concat([bridge_1, bridge_2], ignore_index=True)

# seniors_interpolated_df = temporal_interpolation_with_decay(df_long)

# save_to_file = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/seniors_temporal_interpolation.csv")
# seniors_interpolated_df.to_csv(save_to_file)

# seniors_interpolated_df

In [57]:
def temporal_interpolation_with_decay(df_long):
    # 1. Identify all unique subzones across the entire dataset
    subzones = df_long[['subzone_n', 'pln_area_n']].drop_duplicates()
    
    # --- Segment A: 2010 to 2014 (65 & Over) ---
    # Prepare 2010 and 2015 anchors to define the slope
    a10 = df_long[(df_long['Year'] == 2010) & (df_long['AgeGroup'] == '65 & Over')]
    
    cols_65p = ['65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over']
    a15_65p = df_long[(df_long['Year'] == 2015) & (df_long['AgeGroup'].isin(cols_65p))].copy()
    a15_65p = a15_65p.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].sum().reset_index()
    a15_65p['AgeGroup'], a15_65p['Year'] = '65 & Over', 2015

    # backbone updated to 2010-2014
    # np.arange(2010, 2015) produces [2010, 2011, 2012, 2013, 2014]
    bridge_1 = subzones.assign(k=1).merge(pd.DataFrame({'Year': np.arange(2010, 2015), 'k': 1}), on='k').drop('k', axis=1)
    bridge_1['AgeGroup'] = '65 & Over'
    
    # We still need the 2015 anchor to calculate the 2011-2014 slope correctly
    anchors_1 = pd.concat([a10, a15_65p])
    
    # Create a temporary dataframe for interpolation that includes 2015
    temp_bridge_1 = subzones.assign(k=1).merge(pd.DataFrame({'Year': np.arange(2010, 2016), 'k': 1}), on='k').drop('k', axis=1)
    temp_bridge_1['AgeGroup'] = '65 & Over'
    temp_bridge_1 = temp_bridge_1.merge(anchors_1, on=['subzone_n', 'pln_area_n', 'Year', 'AgeGroup'], how='left')
    
    # Force 0s at anchors
    temp_bridge_1.loc[temp_bridge_1['Year'].isin([2010, 2015]), 'WeightingRatio'] = temp_bridge_1.loc[temp_bridge_1['Year'].isin([2010, 2015]), 'WeightingRatio'].fillna(0)
    
    temp_bridge_1['WeightingRatio'] = temp_bridge_1.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].transform(lambda x: x.interpolate(method='linear'))
    
    # Filter back to only 2010-2014 to avoid overlapping with Segment B
    bridge_1 = temp_bridge_1[temp_bridge_1['Year'] < 2015].copy()

    # --- Segment B: 2015 to 2020 (85 & Over) ---
    a15_85p = df_long[(df_long['Year'] == 2015) & (df_long['AgeGroup'] == '85 & Over')]
    
    cols_85p = ['85 - 89', '90 & over']
    a20_85p = df_long[(df_long['Year'] == 2020) & (df_long['AgeGroup'].isin(cols_85p))].copy()
    a20_85p = a20_85p.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].sum().reset_index()
    a20_85p['AgeGroup'], a20_85p['Year'] = '85 & Over', 2020

    # backbone for 2015-2020
    bridge_2 = subzones.assign(k=1).merge(pd.DataFrame({'Year': np.arange(2015, 2021), 'k': 1}), on='k').drop('k', axis=1)
    bridge_2['AgeGroup'] = '85 & Over'
    
    anchors_2 = pd.concat([a15_85p, a20_85p])
    bridge_2 = bridge_2.merge(anchors_2, on=['subzone_n', 'pln_area_n', 'Year', 'AgeGroup'], how='left')
    
    bridge_2.loc[bridge_2['Year'].isin([2015, 2020]), 'WeightingRatio'] = bridge_2.loc[bridge_2['Year'].isin([2015, 2020]), 'WeightingRatio'].fillna(0)
    
    bridge_2['WeightingRatio'] = bridge_2.groupby(['subzone_n', 'pln_area_n'])['WeightingRatio'].transform(lambda x: x.interpolate(method='linear'))

    return pd.concat([bridge_1, bridge_2], ignore_index=True)

seniors_interpolated_df = temporal_interpolation_with_decay(df_long)

save_to_file = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/seniors_temporal_interpolation.csv")
seniors_interpolated_df.to_csv(save_to_file)

seniors_interpolated_df

,subzone_n,pln_area_n,Year,AgeGroup,WeightingRatio
0,woodlands regional centre,woodlands,2010,65 & Over,0.0
1,woodlands regional centre,woodlands,2011,65 & Over,0.0
2,woodlands regional centre,woodlands,2012,65 & Over,0.0
3,woodlands regional centre,woodlands,2013,65 & Over,0.0
4,woodlands regional centre,woodlands,2014,65 & Over,0.0
...,...,...,...,...,...
4186,murai,western water catchment,2016,85 & Over,0.0
4187,murai,western water catchment,2017,85 & Over,0.0
4188,murai,western water catchment,2018,85 & Over,0.0
4189,murai,western water catchment,2019,85 & Over,0.0


In [58]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup the output directory
output_dir = Path("sanity_check/65&85_year_old_above_checks")
output_dir.mkdir(parents=True, exist_ok=True)

def plot_aggregated_trends(df, subzone_name, pln_area):
    """
    Plots the annual trend of aggregated senior groups to check for smoothness.
    """
    # 1. Filter for the specific subzone
    subset = df[(df['subzone_n'] == subzone_name) & (df['pln_area_n'] == pln_area)]
    
    # 2. Create the plot for '65 & Over' (The 2010-2015 bridge)
    data_65p = subset[subset['AgeGroup'] == '65 & Over'].sort_values('Year')
    
    if not data_65p.empty:
        plt.figure(figsize=(10, 6))
        
        # Plot the interpolated line
        sns.lineplot(data=data_65p, x='Year', y='WeightingRatio', marker='o', label='Interpolated Ratio')
        
        # Highlight the Anchor Years (2010 and 2015)
        anchors = data_65p[data_65p['Year'].isin([2010, 2015])]
        plt.scatter(anchors['Year'], anchors['WeightingRatio'], color='red', s=100, zorder=5, label='Anchor Points')
        
        plt.title(f"Temporal Trend: 65 & Over in {subzone_name}")
        plt.xlabel("Year")
        plt.ylabel("Weighting Ratio")
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        
        plt.savefig(output_dir / f"{subzone_name}_65p_trend.png")
        plt.close()

    # 3. Create the plot for '85 & Over' (The 2015-2020 bridge)
    data_85p = subset[subset['AgeGroup'] == '85 & Over'].sort_values('Year')
    
    if not data_85p.empty:
        plt.figure(figsize=(10, 6))
        sns.lineplot(data=data_85p, x='Year', y='WeightingRatio', marker='s', color='orange', label='Interpolated Ratio')
        
        # Highlight the Anchor Years (2015 and 2020)
        anchors_85 = data_85p[data_85p['Year'].isin([2015, 2020])]
        plt.scatter(anchors_85['Year'], anchors_85['WeightingRatio'], color='black', s=100, zorder=5, label='Anchor Points')
        
        plt.title(f"Temporal Trend: 85 & Over in {subzone_name}")
        plt.xlabel("Year")
        plt.ylabel("Weighting Ratio")
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        
        plt.savefig(output_dir / f"{subzone_name}_85p_trend.png")
        plt.close()

# Run for a few samples
sample_list = seniors_interpolated_df[['subzone_n', 'pln_area_n']].drop_duplicates().head(100)
for _, row in sample_list.iterrows():
    plot_aggregated_trends(seniors_interpolated_df, row['subzone_n'], row['pln_area_n'])

print(f"Aggregated trend plots saved to {output_dir}")

Aggregated trend plots saved to sanity_check/65&85_year_old_above_checks


In [59]:
def interpolate_all_granular_groups(df_long):
    # 1. Setup identifiers and years
    subzones = df_long[['subzone_n', 'pln_area_n']].drop_duplicates()
    years = np.arange(2010, 2021)
    
    # 2. Define Age Group Sets
    # Group A: Common across 2010, 2015, and 2020 (0-4 through 60-64)
    common_groups = [
        '0 - 4', '5 - 9', '10 - 14', '15 - 19', '20 - 24', '25 - 29', 
        '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59', '60 - 64'
    ]
    
    # Group B: Common only across 2015 and 2020 (65-69 through 80-84)
    seniors_2015_2020 = ['65 - 69', '70 - 74', '75 - 79', '80 - 84']

    # --- PROCESS COMMON GROUPS (2010 -> 2020) ---
    backbone_common = subzones.assign(k=1).merge(pd.DataFrame({'Year': years, 'k': 1}), on='k')
    backbone_common = backbone_common.assign(k=1).merge(pd.DataFrame({'AgeGroup': common_groups, 'k': 1}), on='k').drop('k', axis=1)
    
    # Merge anchors and force 0s at 2010, 2015, 2020 to ensure decay/growth trends
    anchors_common = df_long[df_long['AgeGroup'].isin(common_groups)]
    interp_common = backbone_common.merge(anchors_common, on=['subzone_n', 'pln_area_n', 'Year', 'AgeGroup'], how='left')
    
    anchor_years = [2010, 2015, 2020]
    interp_common.loc[interp_common['Year'].isin(anchor_years), 'WeightingRatio'] = interp_common.loc[interp_common['Year'].isin(anchor_years), 'WeightingRatio'].fillna(0)
    
    # Linear interpolation across the whole 2010-2020 period
    interp_common = interp_common.sort_values(['subzone_n', 'pln_area_n', 'AgeGroup', 'Year'])
    interp_common['WeightingRatio'] = interp_common.groupby(['subzone_n', 'pln_area_n', 'AgeGroup'])['WeightingRatio'].transform(lambda x: x.interpolate(method='linear'))

    # --- PROCESS SENIOR GROUPS (2015 -> 2020) ---
    # These only exist as separate brackets from 2015 onwards
    years_senior = np.arange(2015, 2021)
    backbone_senior = subzones.assign(k=1).merge(pd.DataFrame({'Year': years_senior, 'k': 1}), on='k')
    backbone_senior = backbone_senior.assign(k=1).merge(pd.DataFrame({'AgeGroup': seniors_2015_2020, 'k': 1}), on='k').drop('k', axis=1)
    
    anchors_senior = df_long[df_long['AgeGroup'].isin(seniors_2015_2020)]
    interp_senior = backbone_senior.merge(anchors_senior, on=['subzone_n', 'pln_area_n', 'Year', 'AgeGroup'], how='left')
    
    # Force 0s at 2015 and 2020 anchors
    interp_senior.loc[interp_senior['Year'].isin([2015, 2020]), 'WeightingRatio'] = interp_senior.loc[interp_senior['Year'].isin([2015, 2020]), 'WeightingRatio'].fillna(0)
    
    interp_senior = interp_senior.sort_values(['subzone_n', 'pln_area_n', 'AgeGroup', 'Year'])
    interp_senior['WeightingRatio'] = interp_senior.groupby(['subzone_n', 'pln_area_n', 'AgeGroup'])['WeightingRatio'].transform(lambda x: x.interpolate(method='linear'))

    # 3. Combine with your previous '65 & Over' and '85 & Over' bridges
    return pd.concat([interp_common, interp_senior], ignore_index=True)

# Generate full granular dataset
df_granular_interpolated = interpolate_all_granular_groups(df_long)

save_to_file = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/granular_temporal_interpolation.csv")
df_granular_interpolated.to_csv(save_to_file)

df_granular_interpolated

,subzone_n,pln_area_n,Year,AgeGroup,WeightingRatio
0,admiralty,sembawang,2010,0 - 4,3.602458
1,admiralty,sembawang,2011,0 - 4,3.557145
2,admiralty,sembawang,2012,0 - 4,3.511832
3,admiralty,sembawang,2013,0 - 4,3.466519
4,admiralty,sembawang,2014,0 - 4,3.421206
...,...,...,...,...,...
63622,yunnan,jurong west,2016,80 - 84,3.102786
63623,yunnan,jurong west,2017,80 - 84,3.195108
63624,yunnan,jurong west,2018,80 - 84,3.287430
63625,yunnan,jurong west,2019,80 - 84,3.379752


In [32]:
granular_output_dir = Path("sanity_check/granular_age_group_checks")
granular_output_dir.mkdir(parents=True, exist_ok=True)

def plot_granular_trends(df, subzone_name, pln_area):
    """
    Plots annual trends for all granular age groups in a subzone.
    """
    # Filter for the specific subzone
    subset = df[(df['subzone_n'] == subzone_name) & (df['pln_area_n'] == pln_area)]
    
    if subset.empty:
        return

    # Create the plot
    plt.figure(figsize=(14, 8))
    
    # Use a color palette to distinguish age groups
    palette = sns.color_palette("husl", len(subset['AgeGroup'].unique()))
    
    # Plot interpolated lines for all groups
    sns.lineplot(
        data=subset, 
        x='Year', 
        y='WeightingRatio', 
        hue='AgeGroup', 
        marker='o', 
        palette=palette,
        alpha=0.7
    )
    
    # Highlight Anchor Years (2010, 2015, 2020) with larger dots
    # This helps verify if the interpolation is correctly hitting the data points
    anchors = subset[subset['Year'].isin([2010, 2015, 2020])]
    plt.scatter(
        anchors['Year'], 
        anchors['WeightingRatio'], 
        color='black', 
        s=40, 
        zorder=5, 
        label='Anchor Years'
    )
    
    plt.title(f"Granular Age Group Trends: {subzone_name} ({pln_area})")
    plt.xlabel("Year")
    plt.ylabel("Weighting Ratio")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Age Groups')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    # Save the file
    safe_name = subzone_name.replace("/", "_")
    plt.savefig(granular_output_dir / f"{safe_name}_granular_trend.png")
    plt.close()

# 2. Run for sample subzones
# We use head(20) to avoid generating too many files at once
sample_list = df_granular_interpolated[['subzone_n', 'pln_area_n']].drop_duplicates().head(100)

for _, row in sample_list.iterrows():
    plot_granular_trends(df_granular_interpolated, row['subzone_n'], row['pln_area_n'])

print(f"Granular trend plots saved to {granular_output_dir}")

Granular trend plots saved to sanity_check/granular_age_group_checks


In [60]:
## vertically combine the seniors and granular interpolated density ratio dataframes
# 1. Vertically stack the two dataframes
# We use axis=0 (default) to stack them on top of each other
df_final_interpolated_ratios = pd.concat([
    seniors_interpolated_df, 
    df_granular_interpolated
], ignore_index=True)

# 2. Sort for better readability
df_final_interpolated_ratios = df_final_interpolated_ratios.sort_values(
    ['pln_area_n', 'subzone_n', 'Year', 'AgeGroup']
)

# 3. Save the master file
save_path = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/final_density_ratio_interpolation.csv")
df_final_interpolated_ratios.to_csv(save_path, index=False)

display(df_final_interpolated_ratios.head())

,subzone_n,pln_area_n,Year,AgeGroup,WeightingRatio
5192,ang mo kio town centre,ang mo kio,2010,0 - 4,0.0
5203,ang mo kio town centre,ang mo kio,2010,10 - 14,0.0
5214,ang mo kio town centre,ang mo kio,2010,15 - 19,0.0
5225,ang mo kio town centre,ang mo kio,2010,20 - 24,0.0
5236,ang mo kio town centre,ang mo kio,2010,25 - 29,0.0


#### Check that the final density ratio dataframe contains all the unique subzone and planning area pair

    2010 to 2014 data should contain all of the subzone and planning area pair along with the following age groups for each pairs
    ['0 - 4', '5 - 9', '10 - 14', '15 - 19', '20 - 24', '25 - 29', 
        '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59', 
        '60 - 64', '65 & Over']

    2015 to 2020 data should contain all of the subzone and planning area pair along with the following age groups for each pairs
    ['0 - 4', '5 - 9', '10 - 14', '15 - 19', '20 - 24', '25 - 29', 
        '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59', 
        '60 - 64', '65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over']

In [61]:
# Extract pairs from each anchor year
pairs_2010 = density_ratio_df_2010[['subzone_n', 'pln_area_n']]
pairs_2015 = density_ratio_df_2015[['subzone_n', 'pln_area_n']]
pairs_2020 = density_ratio_df_2020[['subzone_n', 'pln_area_n']]

# Combine to get the Master List of all unique subzones across the decade
master_subzones = pd.concat([pairs_2010, pairs_2015, pairs_2020]).drop_duplicates()
master_subzones

,subzone_n,pln_area_n
0,woodlands regional centre,woodlands
1,kranji,sungei kadut
2,turf club,sungei kadut
3,nee soon,yishun
4,yishun west,yishun
...,...,...
288,plantation,tengah
289,tengah industrial estate,tengah
311,bahar,western water catchment
312,cleantech,western water catchment


In [64]:
# Search for '65 & Over' rows that exist in the years 2015-2020
extra_rows = df_final_interpolated_ratios.loc[
    (df_final_interpolated_ratios['Year'] >= 2015) & 
    (df_final_interpolated_ratios['AgeGroup'] == '65 & Over')
]

# Display the first few rows to verify
display(extra_rows.head())

# Check how many such rows exist
print(f"Total extra rows found: {len(extra_rows)}")

,subzone_n,pln_area_n,Year,AgeGroup,WeightingRatio


Total extra rows found: 0


Linear Extrapolation will be used for the density ratio of 2021

In [66]:
def extrapolate_to_2021(df_final):
    # 1. Isolate the last two years to find the trend
    df_2019 = df_final[df_final['Year'] == 2019].copy()
    df_2020 = df_final[df_final['Year'] == 2020].copy()
    
    # Rename columns for calculation
    df_2019 = df_2019.rename(columns={'WeightingRatio': 'Ratio_2019'})
    df_2020 = df_2020.rename(columns={'WeightingRatio': 'Ratio_2020'})
    
    # 2. Merge to calculate the annual change
    trend_df = pd.merge(
        df_2020[['subzone_n', 'pln_area_n', 'AgeGroup', 'Ratio_2020']],
        df_2019[['subzone_n', 'pln_area_n', 'AgeGroup', 'Ratio_2019']],
        on=['subzone_n', 'pln_area_n', 'AgeGroup'],
        how='left'
    )
    
    # Annual Change = 2020 value - 2019 value
    trend_df['Annual_Change'] = trend_df['Ratio_2020'] - trend_df['Ratio_2019']
    
    # 3. Create 2021 Data
    df_2021 = trend_df.copy()
    df_2021['WeightingRatio'] = df_2021['Ratio_2020'] + df_2021['Annual_Change']
    
    # Clip at 0 (Weighting ratios cannot be negative)
    df_2021['WeightingRatio'] = df_2021['WeightingRatio'].clip(lower=0)
    
    # Final cleanup to match original format
    df_2021['Year'] = 2021
    df_2021 = df_2021[['subzone_n', 'pln_area_n', 'Year', 'AgeGroup', 'WeightingRatio']]
    
    # 4. Append back to master dataframe
    return pd.concat([df_final, df_2021], ignore_index=True)

# Run extrapolation
df_master_with_2021 = extrapolate_to_2021(df_final_interpolated_ratios)

save_path = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/subzone_density_ratios/final_density_ratio_interpolation.csv")
df_master_with_2021.to_csv(save_path, index=False)

df_master_with_2021

,subzone_n,pln_area_n,Year,AgeGroup,WeightingRatio
0,ang mo kio town centre,ang mo kio,2010,0 - 4,0.000000
1,ang mo kio town centre,ang mo kio,2010,10 - 14,0.000000
2,ang mo kio town centre,ang mo kio,2010,15 - 19,0.000000
3,ang mo kio town centre,ang mo kio,2010,20 - 24,0.000000
4,ang mo kio town centre,ang mo kio,2010,25 - 29,0.000000
...,...,...,...,...,...
74671,yishun west,yishun,2021,65 - 69,8.215283
74672,yishun west,yishun,2021,70 - 74,6.939292
74673,yishun west,yishun,2021,75 - 79,7.273863
74674,yishun west,yishun,2021,80 - 84,6.847102


In [70]:
def verify_completeness(df_interpolated, master_subzones):
    # Define your required age group lists
    age_groups_2010_2014 = [
        '0 - 4', '5 - 9', '10 - 14', '15 - 19', '20 - 24', '25 - 29', 
        '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59', 
        '60 - 64', '65 & Over'
    ]
    
    age_groups_2015_2020 = [
        '0 - 4', '5 - 9', '10 - 14', '15 - 19', '20 - 24', '25 - 29', 
        '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59', 
        '60 - 64', '65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over'
    ]

    # Check 2010-2015 Segment
    for year in range(2010, 2015):
        year_data = df_interpolated[df_interpolated['Year'] == year]
        actual_pairs = year_data[['subzone_n', 'pln_area_n']].drop_duplicates().shape[0]
        actual_groups = year_data['AgeGroup'].unique()
        
        missing_groups = set(age_groups_2010_2014) - set(actual_groups)
        
        print(f"Year {year}: Subzones: {actual_pairs}/{len(master_subzones)} | Missing Groups: {missing_groups}")

    # Check 2016-2020 Segment
    for year in range(2015, 2022):
        year_data = df_interpolated[df_interpolated['Year'] == year]
        actual_pairs = year_data[['subzone_n', 'pln_area_n']].drop_duplicates().shape[0]
        actual_groups = year_data['AgeGroup'].unique()
        
        missing_groups = set(age_groups_2015_2020) - set(actual_groups)
        
        print(f"Year {year}: Subzones: {actual_pairs}/{len(master_subzones)} | Missing Groups: {missing_groups}")

# Run the check
verify_completeness(df_master_with_2021, master_subzones)

Year 2010: Subzones: 381/381 | Missing Groups: set()
Year 2011: Subzones: 381/381 | Missing Groups: set()
Year 2012: Subzones: 381/381 | Missing Groups: set()
Year 2013: Subzones: 381/381 | Missing Groups: set()
Year 2014: Subzones: 381/381 | Missing Groups: set()
Year 2015: Subzones: 381/381 | Missing Groups: set()
Year 2016: Subzones: 381/381 | Missing Groups: set()
Year 2017: Subzones: 381/381 | Missing Groups: set()
Year 2018: Subzones: 381/381 | Missing Groups: set()
Year 2019: Subzones: 381/381 | Missing Groups: set()
Year 2020: Subzones: 381/381 | Missing Groups: set()
Year 2021: Subzones: 381/381 | Missing Groups: set()


In [72]:
def check_for_extra_age_groups(df_interpolated):
    # Define your expected age group lists
    expected_2010_2014 = [
        '0 - 4', '5 - 9', '10 - 14', '15 - 19', '20 - 24', '25 - 29', 
        '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59', 
        '60 - 64', '65 & Over'
    ]
    
    expected_2015_2021 = [
        '0 - 4', '5 - 9', '10 - 14', '15 - 19', '20 - 24', '25 - 29', 
        '30 - 34', '35 - 39', '40 - 44', '45 - 49', '50 - 54', '55 - 59', 
        '60 - 64', '65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 & Over'
    ]

    # Check Segment 1: 2010 - 2015
    actual_2010_2014 = set(df_interpolated[df_interpolated['Year'] <= 2014]['AgeGroup'].unique())
    extra_groups_1 = actual_2010_2014 - set(expected_2010_2014)
    
    if extra_groups_1:
        print(f"Found extra groups in 2010-2014: {extra_groups_1}")
    else:
        print("2010-2014 Age Groups are clean.")

    # Check Segment 2: 2015 - 2021
    actual_2015_2021 = set(df_interpolated[df_interpolated['Year'] >= 2015]['AgeGroup'].unique())
    extra_groups_2 = actual_2015_2021 - set(expected_2015_2021)
    
    if extra_groups_2:
        print(f"Found extra groups in 2015-2021: {extra_groups_2}")
    else:
        print("2015-2021 Age Groups are clean.")

# Execute the check
check_for_extra_age_groups(df_final_interpolated_ratios)

2010-2014 Age Groups are clean.
2015-2021 Age Groups are clean.


# Step 4: Interpolation Step

In [ ]:
def perform_step_1_and_2():
    
    # dictionary of the available demographic data years and their corresponding subzone data years
    anchor_years = {2010: 2008,
                    2015: 2014,
                    2020: 2019}

    for demographic_year, subzone_year in anchor_years.items():
        subzone_df = get_subzone_data(subzone_year)
        subzone_df_meters = convert_to_geodataframe(subzone_df)

        # calculate area
        subzone_df['area_sqm'] = subzone_df_meters.geometry.area
        subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000
        subzone_areas = subzone_df[['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', "geometry"]].copy()

        demographic_df = get_demographic_data(demographic_year)
        density_df = calculate_density(demographic_df, subzone_areas, demographic_year)
        # fill NaN results with 0 (0 population data for that subzone)
        density_df = density_df.fillna(np.nan)

        print(f"\nPerforming sanity check for {demographic_year}")
        sanity_checks(density_df, subzone_areas, demographic_df)

        national_age_grp_density_df = calculate_national_average_density(density_df, demographic_year)
        density_ratio_df = calculate_density_ratio(density_df, national_age_grp_density_df)
        
        save_to_file = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/subzone_density_ratios/pop_density_ratio_{demographic_year}.csv")
        density_ratio_df.to_csv(save_to_file)

In [ ]:
perform_step_1_and_2()